In [ ]:
import pandas as pd

In [ ]:
import sys

In [ ]:
sys.path.append('..')

In [ ]:
df_questions = pd.read_csv('questions.csv')
df_questions.head()

In [ ]:
row = df_questions.iloc[0]
row.question

In [ ]:

from tools import create_documentation_tools_cached
from doc_agent import create_agent, DocumentationAgentConfig

tools = create_documentation_tools_cached()
agent_config = DocumentationAgentConfig()

agent = create_agent(agent_config, tools)

In [ ]:
from doc_agent import AgentStreamRunner
from jaxn import JSONParserHandler

async def run_agent(agent, user_prompt, message_history=None):
    runner = AgentStreamRunner(agent, JSONParserHandler())
    result = await runner.run(user_prompt, message_history)
    return result

In [ ]:
result = await run_agent(agent, row.question)

In [ ]:

from tests.cost_tracker import calculate_cost


usage = result.usage()

cost = calculate_cost(
    model_name=agent_config.model,
    input_tokens=usage.input_tokens,
    output_tokens=usage.output_tokens
)

In [ ]:
import dataclasses
from tests.utils import collect_tools

tools = collect_tools(result.all_messages())
tools_dicts = [dataclasses.asdict(t) for t in tools]

In [ ]:
rag_output = result.output

In [ ]:
output = {
    'input': row.to_dict(),
    'rag_response': rag_output.model_dump(),
    'tools': tools_dicts,
    'cost': cost
}

In [ ]:
async def run_eval_round(row):
    result = await run_agent(agent, row.question)

    usage = result.usage()

    cost = calculate_cost(
        model_name=agent_config.model,
        input_tokens=usage.input_tokens,
        output_tokens=usage.output_tokens
    )

    tools = collect_tools(result.all_messages())
    tools_dicts = [dataclasses.asdict(t) for t in tools]
    
    rag_output = result.output

    output = {
        'input': row.to_dict(),
        'rag_response': rag_output.model_dump(),
        'tools': tools_dicts,
        'cost': cost
    }

    return output

In [ ]:
output = await run_eval_round(row)

In [ ]:
output

In [ ]:
from tqdm.auto import tqdm

In [ ]:
import traceback

outputs = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    try:
        output = await run_eval_round(row)
        outputs.append(output)

    except Exception as e:
        print(f"Cannot process row {row}: {e}")
        traceback.print_exc()

In [ ]:
len(outputs)

In [ ]:
import json

In [ ]:
with open('evals_run_2026_03_06.json', 'wt') as f_out:
    json.dump(outputs, f_out, indent=2)

In [ ]:
total_cost = sum([o['cost'] for o in outputs])
print(f'total cost: {total_cost:0.3f}')